# Sign Your Way: train ASL + ISL word models
Run the cells top to bottom (Runtime > Run all is fine after you fill in cell 2). Takes roughly 30-90 minutes. **Do not share this notebook after pasting your Kaggle key.**

In [ ]:
!pip -q install -U kagglehub mediapipe opencv-python-headless scikit-learn onnx

In [ ]:
import os
# Paste the long token from Kaggle (it usually starts with KGAT_) between the quotes
os.environ['KAGGLE_API_TOKEN']='PASTE_YOUR_TOKEN_HERE'
# Only for old-style kaggle.json files (username + key). Leave as is if you use the token above.
#os.environ['KAGGLE_USERNAME']='PASTE_YOUR_USERNAME'
#os.environ['KAGGLE_KEY']='PASTE_YOUR_KEY'

In [ ]:
%%writefile extract_landmarks.py
"""Videos -> fixed-length landmark sequences (uses the same MediaPipe hand model as the website).
Usage: python extract_landmarks.py manifest.csv out_dir
manifest.csv columns: path,label,signer"""
import sys, csv, os, urllib.request, numpy as np, cv2, mediapipe as mp
from mediapipe.tasks import python as mpt
from mediapipe.tasks.python import vision
T = 30
MODEL = "hand_landmarker.task"
if not os.path.exists(MODEL):
    urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task", MODEL)
OPTS = vision.HandLandmarkerOptions(base_options=mpt.BaseOptions(model_asset_path=MODEL),
                                    running_mode=vision.RunningMode.VIDEO, num_hands=2)

def norm(v):                       # position/scale independent; identical to the website
    for i in range(2):
        if v[i].any():
            v[i] -= v[i][0]; v[i] /= (np.abs(v[i]).max() + 1e-6)
    return v.reshape(-1)           # 126 features

def sequence(path):
    cap = cv2.VideoCapture(path); fps = cap.get(cv2.CAP_PROP_FPS) or 25
    frames, i = [], 0
    with vision.HandLandmarker.create_from_options(OPTS) as lm:
        while True:
            ok, f = cap.read()
            if not ok: break
            img = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
            r = lm.detect_for_video(img, int(i * 1000 / fps)); i += 1
            v = np.zeros((2, 21, 3), np.float32)
            hs = sorted(([[q.x, q.y, q.z] for q in h] for h in r.hand_landmarks), key=lambda a: a[0][0])
            for k, a in enumerate(hs[:2]): v[k] = a      # hands ordered left-to-right in the image
            frames.append(norm(v))
    frames = [f for f in frames if f.any()]             # drop frames without hands (the website does too)
    if len(frames) < 8: return None
    return np.stack([frames[k] for k in np.linspace(0, len(frames) - 1, T).astype(int)])

if __name__ == "__main__":
    rows = list(csv.DictReader(open(sys.argv[1]))); out = sys.argv[2]; os.makedirs(out, exist_ok=True)
    X, y, g, skipped = [], [], [], 0
    for n, r in enumerate(rows):
        try: s = sequence(r["path"])
        except Exception as e: s = None; print("skip", r["path"], e)
        if s is None: skipped += 1; continue
        X.append(s); y.append(r["label"]); g.append(r["signer"])
        if n % 100 == 0: print(f"{n}/{len(rows)} videos done", flush=True)
    np.savez(f"{out}/data.npz", X=np.array(X), y=np.array(y), g=np.array(g))
    print(f"saved {len(X)} sequences, skipped {skipped}")


In [ ]:
%%writefile train.py
"""Train + honestly evaluate one language model. Usage: python train.py data/asl/data.npz models/asl
Train and test signers never overlap. Writes model.onnx, labels.json, eval_report.json."""
import sys, os, json, numpy as np, torch, torch.nn as nn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix

d = np.load(sys.argv[1]); out = sys.argv[2]; os.makedirs(out, exist_ok=True)
labels = sorted(set(d["y"])); idx = {l: i for i, l in enumerate(labels)}
X, y, g = d["X"].astype(np.float32), np.array([idx[l] for l in d["y"]]), d["g"]
tr, te = next(GroupShuffleSplit(1, test_size=.25, random_state=0).split(X, y, g))
assert not set(g[tr]) & set(g[te]), "signer leakage"

class Net(nn.Module):
    def __init__(s, n):
        super().__init__(); s.gru = nn.GRU(126, 128, 2, batch_first=True, dropout=.3, bidirectional=True)
        s.fc = nn.Sequential(nn.Dropout(.3), nn.Linear(256, n))
    def forward(s, x): return s.fc(s.gru(x)[0].mean(1))

m, opt = Net(len(labels)), None
opt = torch.optim.AdamW(m.parameters(), 1e-3, weight_decay=1e-2)
Xt, yt = torch.tensor(X[tr]), torch.tensor(y[tr])
for ep in range(60):
    m.train(); p = torch.randperm(len(Xt))
    for i in range(0, len(p), 32):
        b = p[i:i+32]; xb = Xt[b] + .01 * torch.randn_like(Xt[b])      # light noise augmentation
        loss = nn.functional.cross_entropy(m(xb), yt[b], label_smoothing=.1); opt.zero_grad(); loss.backward(); opt.step()
m.eval()
with torch.no_grad(): pred = m(torch.tensor(X[te])).argmax(1).numpy()
rep = classification_report(y[te], pred, labels=range(len(labels)), target_names=labels, output_dict=True, zero_division=0)
json.dump({"n_train": len(tr), "n_test": len(te), "test_signers": sorted(set(g[te])), "report": rep,
           "confusion": confusion_matrix(y[te], pred, labels=range(len(labels))).tolist()}, open(f"{out}/eval_report.json", "w"), indent=1)
json.dump(labels, open(f"{out}/labels.json", "w"))
torch.onnx.export(m, torch.zeros(1, 30, 126), f"{out}/model.onnx", input_names=["seq"], output_names=["logits"], dynamic_axes={"seq": {0: "b"}}, dynamo=False)
print("test accuracy:", rep["accuracy"], "(signers held out:", len(set(g[te])), ")")


In [ ]:
# Downloads the datasets straight from Kaggle using your token. No Google Drive needed.
import kagglehub, os
def get(name, slug):
    return kagglehub.dataset_download(slug)

## ASL (WLASL). Picks the 50 words with the most clips

In [ ]:
import kagglehub, json, os, glob, csv, collections
p=get('wlasl','risangbaskoro/wlasl-processed'); print(p, os.listdir(p))
meta=json.load(open(glob.glob(p+'/**/WLASL_v0.3.json',recursive=True)[0]))
vids={os.path.basename(f)[:-4]:f for f in glob.glob(p+'/**/*.mp4',recursive=True)}
rows=[(g['gloss'],i['video_id'],i['signer_id']) for g in meta for i in g['instances'] if i['video_id'] in vids]
N=50  # number of words; lower = more accurate
keep={g for g,_ in collections.Counter(r[0] for r in rows).most_common(N)}
with open('asl.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['path','label','signer'])
    for g,v,s in rows:
        if g in keep: w.writerow([vids[v],g,s])
print(len(keep),'words;',sum(r[0] in keep for r in rows),'clips')

In [ ]:
!python extract_landmarks.py asl.csv data/asl
!python train.py data/asl/data.npz models/asl

## ISL (prasadshet dataset)
This assumes one folder per word. Check the printout. If the clips are not organised that way, tell me what the folders look like and I will adjust.

In [ ]:
p=get('isl','prasadshet/indian-sign-language-video-dataset'); print(p, os.listdir(p))
files=[f for f in glob.glob(p+'/**/*',recursive=True) if f.lower().endswith(('.mp4','.avi','.mov','.mkv'))]
lab=lambda f:os.path.basename(os.path.dirname(f))
cnt=collections.Counter(lab(f) for f in files); print(len(files),'videos',len(cnt),'labels'); print(cnt.most_common(15))
keep={l for l,c in cnt.most_common(50) if c>=5}
with open('isl.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['path','label','signer'])
    for x in files:
        if lab(x) in keep: w.writerow([x,lab(x),os.path.basename(x)])  # signer unknown: accuracy will look BETTER than real

In [ ]:
!python extract_landmarks.py isl.csv data/isl
!python train.py data/isl/data.npz models/isl

## Download. Upload the `models` folder to GitHub next to index.html

In [ ]:
!zip -qr models.zip models
from google.colab import files; files.download('models.zip')